# Test the same 10 photos three ways

This notebook uses the checkpoint from training and the same seed-42 split as the training notebook: 24,800 train images and 200 test images on the Colab disk.

The same 10 test photos are shown three times:

1. **Plain.** No watermark. The decoder should say the photo is not watermarked.
2. **Watermark only.** The encoder hides the message `genAI`. No blur, crop, JPEG, or color change.
3. **Watermark plus distortion.** The same stamped photo is then blurred, recolored, JPEG-compressed, and cropped the way training does.

A photo counts as watermarked only when the check bits accept the packet (`bch_ok`). That is the same yes/no used in training. Bit accuracy is how many of the `genAI` bits matched. On a plain photo that number is not a score, because those bits were never hidden there.


In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install -q torch torchvision
%pip install -q lpips opencv-python-headless pillow bchlib matplotlib


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:
    print("Drive not mounted (%s)." % error)

CANDIDATES = [
    Path("/content/drive/MyDrive/own"),
    Path("/content/own"),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT_ROOT = None
for path in CANDIDATES:
    if (path / "stegwatermark").is_dir():
        PROJECT_ROOT = path
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Upload the own folder (the directory that contains stegowatermark/).")
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


## Extract train and test

Same split as the training notebook. The zip stays on Drive. The images are written to `/content/data/train` and `/content/data/test`. Seed 42 keeps the same 200 photos in the test folder every time. If those folders are already full, this cell skips the unzip.


In [ ]:
import os
import random
import shutil
import zipfile

DRIVE_ZIP = "/content/drive/MyDrive/own/datasets/mirflickr25k.zip"
LOCAL_EXTRACT = "/content/mirflickr_raw"
LOCAL_TRAIN = "/content/data/train"
LOCAL_TEST = "/content/data/test"

def jpeg_count(folder):
    if not os.path.isdir(folder):
        return 0
    return sum(name.lower().endswith((".jpg", ".jpeg", ".png")) for name in os.listdir(folder))

if jpeg_count(LOCAL_TRAIN) >= 24800 and jpeg_count(LOCAL_TEST) >= 200:
    print("Local split already present: train %d, test %d" % (jpeg_count(LOCAL_TRAIN), jpeg_count(LOCAL_TEST)))
else:
    if not os.path.exists(DRIVE_ZIP):
        raise FileNotFoundError("Cannot find zip file at %s." % DRIVE_ZIP)
    print("Extracting dataset from Google Drive to local runtime... (takes about a minute)")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        zf.extractall(LOCAL_EXTRACT)
    all_images = []
    for root, dirs, files in os.walk(LOCAL_EXTRACT):
        for name in files:
            if name.lower().endswith((".jpg", ".jpeg", ".png")) and not name.startswith("."):
                all_images.append(os.path.join(root, name))
    all_images.sort()
    print("Total images found in zip: %d" % len(all_images))
    if len(all_images) < 25000:
        raise RuntimeError("Expected 25000 images, found %d. The zip is incomplete." % len(all_images))
    random.seed(42)
    random.shuffle(all_images)
    test_pool = all_images[:200]
    train_pool = all_images[200:]
    os.makedirs(LOCAL_TRAIN, exist_ok=True)
    os.makedirs(LOCAL_TEST, exist_ok=True)
    for img in train_pool:
        dest = os.path.join(LOCAL_TRAIN, os.path.basename(img))
        if not os.path.exists(dest):
            shutil.move(img, dest)
    for img in test_pool:
        dest = os.path.join(LOCAL_TEST, os.path.basename(img))
        if not os.path.exists(dest):
            shutil.move(img, dest)
    print("Local train: %d" % jpeg_count(LOCAL_TRAIN))
    print("Local test: %d" % jpeg_count(LOCAL_TEST))


## Checkpoint and the 10 test photos

`latest.pt` is the file the training notebook saves. The 10 photos are the first 10 names in the test folder, so every run below uses that same list.


In [ ]:
from pathlib import Path

CHECKPOINT = "/content/drive/MyDrive/watermark_checkpoints/latest.pt"
IMAGE_DIR = "/content/data/test"
MESSAGE = "genAI"
N_IMAGES = 10

if not Path(CHECKPOINT).is_file():
    alt = Path("/content/drive/MyDrive/own/watermark_checkpoints/latest.pt")
    if alt.is_file():
        CHECKPOINT = str(alt)
    else:
        raise FileNotFoundError("No checkpoint at %s" % CHECKPOINT)
print("checkpoint", CHECKPOINT)
print("message", MESSAGE)


In [ ]:
from stegowatermark.eval import load_encoder_decoder

encoder, decoder, device, step = load_encoder_decoder(CHECKPOINT)
print("loaded step", step, "on", device)

folder = Path(IMAGE_DIR)
paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if len(paths) < N_IMAGES:
    raise FileNotFoundError("Need %d images in %s, found %d. Run the extract cell first." % (N_IMAGES, IMAGE_DIR, len(paths)))
paths = paths[:N_IMAGES]
print("using", len(paths), "test images")
for path in paths:
    print(" ", path.name)


## Plain, watermarked, and watermarked plus distortion

The secret is the text `genAI`, packed into 100 bits with the same check bits as training. The distorted column uses the training blur, color, JPEG, and 10% border crop at full strength. Nothing is warped in that column, so the comparison is the photo itself versus the same photo after those edits.

The caption under each photo is the decoder's yes/no. Yes means the check bits accepted a message.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from IPython.display import display

from stegowatermark.codec import bits_to_text, text_to_bits
from stegowatermark.distort import photometric, random_border_crop
from stegowatermark.eval import bit_accuracy, embed, load_cover, read_bits
from stegowatermark.train_loop import TrainConfig

bits = text_to_bits(MESSAGE, encoder.secret_size)
damage_cfg = TrainConfig()

def judge(image):
    probabilities = read_bits(decoder, image)
    text, ok = bits_to_text(probabilities.tolist())
    accuracy = bit_accuracy(probabilities, bits)
    return ok, text, accuracy

def apply_training_distortion(stamped, index):
    random.seed(1000 + index)
    torch.manual_seed(1000 + index)
    damaged = photometric(stamped, 10000, damage_cfg)
    damaged = random_border_crop(damaged, 0.10)
    return damaged.clamp(0, 1)

def caption(title, ok, text, accuracy):
    found = "yes" if ok else "no"
    shown = text if text else "—"
    return "%s\nwatermark: %s\ntext: %s\nbit_acc: %.3f" % (title, found, shown, accuracy)

rows = []
figure, axes = plt.subplots(N_IMAGES, 3, figsize=(9, 3.4 * N_IMAGES))
if N_IMAGES == 1:
    axes = axes.reshape(1, 3)

for index, path in enumerate(paths):
    cover = load_cover(path).to(device)
    plain_ok, plain_text, plain_acc = judge(cover)
    stamped, _ = embed(encoder, cover, bits)
    clean_ok, clean_text, clean_acc = judge(stamped)
    damaged = apply_training_distortion(stamped, index)
    hurt_ok, hurt_text, hurt_acc = judge(damaged)

    panels = [
        (cover, caption("plain", plain_ok, plain_text, plain_acc)),
        (stamped, caption("watermark only", clean_ok, clean_text, clean_acc)),
        (damaged, caption("watermark + distortion", hurt_ok, hurt_text, hurt_acc)),
    ]
    for column, (image, title) in enumerate(panels):
        array = image[0].detach().clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        axes[index, column].imshow(array)
        axes[index, column].set_title(title, fontsize=9)
        axes[index, column].axis("off")
    rows.append({
        "image": path.name,
        "plain": plain_ok,
        "plain_text": plain_text,
        "watermark": clean_ok,
        "watermark_text": clean_text,
        "watermark_bit_acc": round(clean_acc, 3),
        "distorted": hurt_ok,
        "distorted_text": hurt_text,
        "distorted_bit_acc": round(hurt_acc, 3),
    })

figure.tight_layout()
display(figure)
plt.close(figure)

plain_hits = sum(row["plain"] for row in rows)
clean_hits = sum(row["watermark"] for row in rows)
hurt_hits = sum(row["distorted"] for row in rows)
print("plain photos called watermarked: %d / %d" % (plain_hits, N_IMAGES))
print("clean watermarks found: %d / %d" % (clean_hits, N_IMAGES))
print("distorted watermarks found: %d / %d" % (hurt_hits, N_IMAGES))
for row in rows:
    print(
        "%s  plain=%s  watermark=%s (%s, %.3f)  distorted=%s (%s, %.3f)"
        % (
            row["image"],
            "yes" if row["plain"] else "no",
            "yes" if row["watermark"] else "no",
            row["watermark_text"],
            row["watermark_bit_acc"],
            "yes" if row["distorted"] else "no",
            row["distorted_text"],
            row["distorted_bit_acc"],
        )
    )


## Accuracy on all 200 test photos

The figure above is 10 photos. This cell scores every image in the test folder. Each number is the average bit match to `genAI` over those photos, not one batch.

Plain photos were never watermarked, so their average should stay near 0.50. Watermarked photos should be high. The distorted column is the same stamps after blur, color, JPEG, and the 10% crop, averaged so one harsh photo does not swing the result.


In [ ]:
import random
from pathlib import Path

import torch

from stegowatermark.codec import bits_to_text, text_to_bits
from stegowatermark.distort import photometric, random_border_crop
from stegowatermark.eval import bit_accuracy, embed, load_cover, read_bits
from stegowatermark.train_loop import TrainConfig

test_paths = sorted(
    path for path in Path(IMAGE_DIR).iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
if len(test_paths) < 200:
    raise FileNotFoundError("Need 200 test images in %s, found %d." % (IMAGE_DIR, len(test_paths)))
test_paths = test_paths[:200]
secret_bits = text_to_bits(MESSAGE, encoder.secret_size)
damage = TrainConfig()

def score(image):
    probabilities = read_bits(decoder, image)
    text, found = bits_to_text(probabilities.tolist())
    return bit_accuracy(probabilities, secret_bits), found, text

plain_acc = []
mark_acc = []
hurt_acc = []
plain_found = 0
mark_found = 0
hurt_found = 0

for index, path in enumerate(test_paths):
    cover = load_cover(path).to(device)
    accuracy, found, _ = score(cover)
    plain_acc.append(accuracy)
    plain_found += int(found)

    stamped, _ = embed(encoder, cover, secret_bits)
    accuracy, found, _ = score(stamped)
    mark_acc.append(accuracy)
    mark_found += int(found)

    random.seed(1000 + index)
    torch.manual_seed(1000 + index)
    distorted = photometric(stamped, 10000, damage)
    distorted = random_border_crop(distorted, 0.10).clamp(0, 1)
    accuracy, found, _ = score(distorted)
    hurt_acc.append(accuracy)
    hurt_found += int(found)

    if (index + 1) % 25 == 0 or index + 1 == len(test_paths):
        print("scored %d / %d" % (index + 1, len(test_paths)))

def mean(values):
    return sum(values) / len(values)

n = len(test_paths)
print("images", n)
print("plain                  bit_acc %.3f   called watermarked %d / %d" % (mean(plain_acc), plain_found, n))
print("watermark only         bit_acc %.3f   called watermarked %d / %d" % (mean(mark_acc), mark_found, n))
print("watermark + distortion bit_acc %.3f   called watermarked %d / %d" % (mean(hurt_acc), hurt_found, n))


## How to read the counts

`plain photos called watermarked` should be 0. A yes there means an ordinary photo was accepted as if it contained `genAI`.

`clean watermarks found` should be 10 if the checkpoint reads an undamaged stamp. The text on those panels should be `genAI`.

`distorted watermarks found` is how many of those same stamps still read after blur, color, JPEG, and the crop. A miss here with a yes in the middle column means the message was written and the damage removed it.


## Finding more distorted watermarks

The 26/200 count uses the check bits, which give up after 5 wrong bits. That is about 0.95. This cell keeps that rule, and also counts a photo as watermarked when at least 85% of the `genAI` bits match. A plain photo matches about half the bits by chance, so 0.85 is still far above a photo with no watermark.

It scores the same harsh damage as the cell above, then a milder damage: less color wash, JPEG no worse than quality 80, and a 5% crop. The message hidden in the photo does not change.


In [ ]:
import random
from pathlib import Path

import torch

from stegowatermark.codec import bits_to_text, text_to_bits
from stegowatermark.distort import photometric, random_border_crop
from stegowatermark.eval import bit_accuracy, embed, load_cover, read_bits
from stegowatermark.train_loop import TrainConfig

FOUND_AT = 0.85
test_paths = sorted(
    path for path in Path(IMAGE_DIR).iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:200]
if len(test_paths) < 200:
    raise FileNotFoundError("Need 200 test images, found %d." % len(test_paths))

secret_bits = text_to_bits(MESSAGE, encoder.secret_size)
harsh = TrainConfig()
mild = TrainConfig(
    sat=0.15,
    jpeg_quality=80,
    bri=0.1,
    hue=0.02,
    noise=0.01,
    contrast_low=0.85,
    contrast_high=1.15,
)

def bit_match(image):
    probabilities = read_bits(decoder, image)
    text, check_ok = bits_to_text(probabilities.tolist())
    accuracy = bit_accuracy(probabilities, secret_bits)
    return accuracy, check_ok

def distort(stamped, index, cfg, crop, seed):
    random.seed(seed + index)
    torch.manual_seed(seed + index)
    damaged = photometric(stamped, 10000, cfg)
    return random_border_crop(damaged, crop).clamp(0, 1)

groups = {
    "plain": [],
    "watermark only": [],
    "harsh distortion": [],
    "mild distortion": [],
}
checks = {name: 0 for name in groups}
above = {name: 0 for name in groups}

for index, path in enumerate(test_paths):
    cover = load_cover(path).to(device)
    stamped, _ = embed(encoder, cover, secret_bits)
    versions = {
        "plain": cover,
        "watermark only": stamped,
        "harsh distortion": distort(stamped, index, harsh, 0.10, 1000),
        "mild distortion": distort(stamped, index, mild, 0.05, 2000),
    }
    for name, image in versions.items():
        accuracy, check_ok = bit_match(image)
        groups[name].append(accuracy)
        checks[name] += int(check_ok)
        above[name] += int(accuracy >= FOUND_AT)
    if (index + 1) % 25 == 0 or index + 1 == len(test_paths):
        print("scored %d / %d" % (index + 1, len(test_paths)))

n = len(test_paths)
print("found when bit_acc >= %.2f, or when the check bits accept the word" % FOUND_AT)
for name in groups:
    mean = sum(groups[name]) / n
    print(
        "%-18s  bit_acc %.3f   bit_acc>=%.2f %d / %d   check bits %d / %d"
        % (name, mean, FOUND_AT, above[name], n, checks[name], n)
    )
